# Notebook 1 - Phase-construction
## Knowledge Graph construction - Overview

This notebook documents the construction of a knowledge graph (KG) to support 
named entity recognition (NER) for Irish, built using the Adkins et al. (2025) 
corpus as the coverage target.

The pipeline covers:
1. Corpus analysis and entity vocabulary extraction
2. Wikidata type mapping and coverage estimation
3. Oireachtas API cross-reference for PER entities
4. Wikidata entity fetch via REST API (PER, ORG, LOC)
5. Stub node creation for unmatched entities
6. Neo4j graph load and validation
7. PyKEEN embedding training and export

**Note:** This notebook is a documented record of work completed locally. Cells that require Neo4j, the Wikidata REST API, or PyKEEN training are not intended to be re-run live. Outputs shown are from the original local runs. The key output files generated by this pipeline (per_nodes.csv, loc_nodes.csv, org_nodes.csv, stub_nodes.csv, kg_triples_clean.tsv, TransE_phase_a_embeddings.pkl) are available in the irish-ner-kg-consolidated Kaggle dataset at michaelmarkey64/irish-ner-kg-consolidated.

## Dependencies

The following packages are required for the cells that are locally executable:

- `pandas` — node CSV loading and coverage diagnostics
- `requests` — Oireachtas API fetch
- `fuzzywuzzy` — Levenshtein distance-based fuzzy matching for PER entity cross-reference
- `collections` — standard library; used for entity vocabulary extraction

The packages below were used during the original build but are not required to read this notebook:

- `neo4j` (Python driver v6.2.0) — graph node and edge loading via Bolt protocol
- `pykeen` — TransE, RotatE, and ComplEx embedding training
- `SPARQLWrapper` — SPARQL endpoint queries; endpoint was unavailable throughout and REST API was used exclusively

## Fuzzy Matching: fuzzywuzzy

fuzzywuzzy is a Python library for fuzzy string matching based on Levenshtein distance — the minimum number of single-character edits (insertions, deletions, substitutions) required to transform one string into another. The `fuzz` module exposes the individual matching functions used in this notebook.

`fuzz.ratio` computes straightforward Levenshtein similarity between two full strings as a percentage. It is sensitive to character-level differences including accents.

`fuzz.partial_ratio` computes the ratio on the best matching substring rather than the full string, which is useful when one string is a subset of the other (e.g. "Heather Humphreys" vs "Humphreys").

`fuzz.token_sort_ratio` sorts the tokens in both strings alphabetically before computing the ratio. This eliminates word order as a source of mismatch and is the primary method used for multi-token Irish names, where surname and given name order varies across sources (e.g. "Ó Cuív Éamon" vs "Éamon Ó Cuív").

`fuzz.token_set_ratio` splits the comparison into the intersection and remainder of the two token sets and takes the best score across several combinations. It handles cases where one string contains additional tokens not present in the other.

In this notebook, `token_sort_ratio` is applied to multi-token names and `fuzz.ratio` to surname-only entities. Thresholds of 88 and 95 respectively were set empirically to balance recall against false positive risk.

In [2]:
import json                           # parsing Wikidata REST API responses
import os                             # file path handling
import pickle                         # loading PyKEEN embedding files
import time                           # rate limiting during API calls

from collections import defaultdict 
import pandas as pd  
import requests
from fuzzywuzzy import fuzz

from IPython.display import display
import warnings
# Suppresses fuzzywuzzy deprecation warning re: python-Levenshtein
warnings.filterwarnings('ignore')

## Corpus Overview

The Adkins et al. (2025) dataset was downloaded from the ner4Irish GitHub 
repository. The splits used throughout this project are:

- `train_final.conll` — training split
- `NER_Irish_validation.conll` — validation split
- `NER_Irish_test.conll` — test split

The corpus uses standard CoNLL BIO format with tab-separated token/tag pairs 
and three entity classes: PER, ORG, and LOC.

### Entity Counts

**Mentions per split:**

| Class | Train mentions | Train unique | Val unique | Test unique |
|---|---|---|---|---|
| PER | 582 | 514 | 61 | 99 |
| ORG | 831 | 555 | 60 | 82 |
| LOC | 686 | 476 | 48 | 100 |

**Unique entities across all splits combined:**

| Class | Unique entities |
|---|---|
| PER | 651 |
| ORG | 656 |
| LOC | 573 |
| **Total** | **1,880** |

This combined vocabulary of 1,880 unique entity strings serves as the
coverage target for the knowledge graph.


In [3]:
def extract_entities(filepath):
    # Parses a CoNLL BIO file and returns unique entity strings per class.
    # B- tags open a span, I- tags extend it, any other tag closes it.
    # Multi-token spans are joined with a single space.
    # Returns a dict of sets: {entity_class: {entity_string, ...}}
    entities = defaultdict(set)
    current_tokens, current_type = [], None
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_tokens:
                    entities[current_type].add(" ".join(current_tokens))
                current_tokens, current_type = [], None
                continue
            parts = line.split()
            token, tag = parts[0], parts[-1]
            if tag.startswith("B-"):
                if current_tokens:
                    entities[current_type].add(" ".join(current_tokens))
                current_tokens = [token]
                current_type = tag[2:]
            elif tag.startswith("I-") and current_tokens:
                current_tokens.append(token)
            else:
                if current_tokens:
                    entities[current_type].add(" ".join(current_tokens))
                current_tokens, current_type = [], None
    return entities

# Local paths — non-reproducible, documented record only.
# Corpus files are available at michaelmarkey64/irish-ner-kg-consolidated under conll/ on Kaggle.
BASE = "/Users/michaelmarkey/Desktop/Local_Daily_Dissertation/Dissertation - 27 05 2026/Train:Test:Val Split/"

train_entities = extract_entities(BASE + "train_final.conll")
val_entities   = extract_entities(BASE + "NER_Irish_validation.conll")
test_entities  = extract_entities(BASE + "NER_Irish_test.conll")

for etype in ["PER", "ORG", "LOC"]:
    all_ents = train_entities[etype] | val_entities[etype] | test_entities[etype]
    print(f"{etype}: {len(all_ents)} unique entities across all splits")

FileNotFoundError: [Errno 2] No such file or directory: '/Users/michaelmarkey/Desktop/Local_Daily_Dissertation/Dissertation - 27 05 2026/Train:Test:Val Split/train_final.conll'

## Unseen Entity Problem

In this corpus, 75% of test entities (210 out of 281) do not appear anywhere in the 
training set while 73% of validation entities (124 out of 169) are similarly unseen in 
training

The model will encounter the majority of test entities with no parametric 
knowledge from fine-tuning alone. The KG provides background knowledge about 
these entities regardless of whether they appeared in training data. 
This is a key factor in motivating the augmentation approach.

## Irish Morphology: The Matching Challenge

Irish initial mutation (lenition and eclipsis) means the same entity surface form varies depending on grammatical context. 214 entities across all classes begin with a lowercase letter as a direct result of this mutation.

Examples observed in the corpus: Baile Átha Cliath appears as Bhaile Átha Cliath (lenited) and mBaile Átha Cliath (eclipsed); Gaeltacht appears as Ghaeltacht and nGaeltacht; Parlaimint na hEorpa appears as Pharlaimint na hEorpa; Conradh na Gaeilge appears as Chonradh na Gaeilge.

During the Wikidata spot-check, the majority of entities marked as not found were not genuinely absent from Wikidata — they were searched in their mutated surface form. After canonicalisation, most resolved correctly.

This has three implications for KG design. Exact string matching against Wikidata labels will fail for a substantial proportion of entities. A canonicalisation step is required before KG lookup. Each KG node should store surface form variants as node properties alongside the canonical label.

## PER Entity Structure

PER entities fall into two structurally distinct groups requiring different KG treatment. The majority are named individuals (e.g. Bertie Ahern, Éamon Ó Cuív, Katie Taylor, Ryan Tubridy), matched against Wikidata Q5 (human) and represented as PER nodes. A smaller group are role titles (e.g. Taoiseach, Tánaiste, Aire Stáit, Leas-Cheann Comhairle), which are tagged as PER in the corpus but represent offices rather than individuals. These are matched against Wikidata Q294414 (public office) and represented as OFFICE nodes, with PER→OFFICE edges derived from P39 on politician entries.

A small number of entities were identified as corpus noise during manual review — fictional characters and common nouns mistakenly tagged as PER in the corpus (e.g. Van Helsing, An Pionta, Tommy an Táilliúra). These were excluded from the graph entirely.


## Wikidata and Oireachtas API: Entity Grounding

### Wikidata Type Mappings

Each entity class was mapped to a Wikidata P31 value and a set of properties to extract. Named individuals (Q5, human) were fetched with P39 (position held), P102 (party), P27 (country of citizenship), and Irish and English labels. Role titles were matched to Q294414 (public office) with P1308 (officeholder) and labels. Organisations were matched to Q7278 (political party) or Q43229 (organisation), filtered by P17=Q27 (Ireland), with headquarters and parent organisation properties where available. Locations were matched to settlement and administrative territory types, filtered by P17=Q27, with Logainm ID (P6872) included where populated.

### False Positive Risk

Short or ambiguous strings produced incorrect matches during spot-checking: `tAire` matched a village in Lebanon, `HSE` matched an education organisation in Moscow, `Dála` matched a chemical compound. All queries were filtered by P17=Q27 and entity type — text search alone was insufficient.

### Estimated Wikidata Coverage

A sample of the top 30 entities per class was checked against the Wikidata Search API. Approximate hit rates: PER around 90%, ORG around 73%, LOC around 63%. Irish label coverage among matched entities was lower, particularly for LOC. These figures likely understate true coverage — most not-found entities were lenited or eclipsed surface forms of entities that do exist in Wikidata under their canonical form.

### Oireachtas API Cross-Reference

The Oireachtas API (api.oireachtas.ie/v1/members) was used as a supplementary source for PER entities, yielding records for all Dáil and Seanad members from the foundation of the state to the present. It provided confirmed Irish-language name spellings and party data that Wikidata lacked for many minor TDs. The corpus PER entities were fuzzy-matched against this list using `token_sort_ratio` for multi-token names and `fuzz.ratio` for surname-only entities, with thresholds of 88 and 95 respectively.

Approximately 11% of named individuals matched, which is consistent with the domain scope of the API — the no-match group includes writers, athletes, journalists, and historical figures outside the political domain. Surname-only matches were flagged as low confidence and linked to multiple candidates rather than hard-linked to a single node. Full-name matches scoring between 88 and 92 were treated as near-misses and not hard-linked without manual verification. Pagination was handled with a 100-record batch loop; a short sleep between requests was applied to avoid overloading the public endpoint.

In [22]:
# Oireachtas API fetch — documented record
# This cell fetches the full Oireachtas member list (1,928 members)
# Output file: oireachtas_members.csv

HEADERS = {
    "User-Agent": "MichaelMarkeyDissertation/1.0 (TU Dublin MSc; michael.markey@tudublin.ie)"
}

all_members = []
limit = 100
offset = 0

while True:
    r = requests.get(
        "https://api.oireachtas.ie/v1/members",
        params={"limit": limit, "skip": offset, "format": "json"},
        headers=HEADERS,
        timeout=15
    )
    data = r.json()
    batch = data["results"]

    if not batch:
        break

    for item in batch:
        m = item["member"]
        party, constituency, house_no = None, None, None
        if m["memberships"]:
            latest = m["memberships"][0]["membership"]
            house_no = latest["house"]["houseNo"]
            if latest["parties"]:
                party = latest["parties"][0]["party"]["showAs"]
            if latest["represents"]:
                constituency = latest["represents"][0]["represent"]["showAs"]
        all_members.append({
            "member_code": m["memberCode"],
            "full_name_en": m["fullName"],
            "first_name": m["firstName"],
            "last_name": m["lastName"],
            "party": party,
            "constituency": constituency,
            "house_no": house_no,
            "uri": m["uri"]
        })

    offset += limit
    if len(batch) < limit:
        break
    time.sleep(0.3)

df_members = pd.DataFrame(all_members)
print(f"Fetched {len(df_members)} members")
print(f"Unique member codes: {df_members['member_code'].nunique()}")

# Original output:
# Fetched 1928 members
# Unique member codes: 1928

Fetched 1928 members
Unique member codes: 1928


## Handling Mutations in Irish

Irish initial mutations (lenition and eclipsis) alter the first letter or letters of a word depending on grammatical context, producing multiple surface forms for the same entity. Rule-based stripping is used here rather than a full morphological analyser — the mutation patterns are finite and regular, full lemmatisation is unnecessary for the specific task of entity string normalisation before Wikidata lookup, and a morphological analyser would introduce an additional runtime dependency. This approach is consistent with preprocessing conventions in existing Irish NLP pipelines (Scannell 2014, Lynn et al. 2019).

Four mutation types are handled. Definite article stripping removes `an`, `na`, `an t-`, and `na h-` prefixes (e.g. `an tOireachtas` → `Oireachtas`). Eclipsis strips prefixes `mb-`, `gc-`, `nd-`, `bhf-`, `ng-`, `bp-`, `dt-`, and vowel-initial `n-` and `h-` prefixes (e.g. `mBaile` → `Baile`, `hÉireann` → `Éireann`). Lenition strips the added `h` from consonant pairs `Bh`, `Ch`, `Dh`, `Fh`, `Gh`, `Mh`, `Ph`, `Sh`, `Th` (e.g. `Chomhairle` → `Comhairle`). Genitive article stripping removes `d'` before vowel-initial strings (e.g. `d'Údarás` → `Údarás`). Eclipsis patterns are checked before lenition to avoid mis-stripping overlapping prefixes such as `bhf-`.

In [4]:
def canonicalise_irish(surface_form):
    s = surface_form.strip()
    
    # Normalise curly apostrophes to straight
    s = s.replace('\u2019', "'").replace('\u2018', "'")

    # Strip leading Irish definite articles
    for article in ('an t-', 'na h-', 'an ', 'na '):
        if s.lower().startswith(article.lower()):
            s = s[len(article):].strip()
            break

    # Strip t- before vowel (an tOireachtas → tOireachtas → Oireachtas)
    if len(s) > 1 and s[0] == 't' and s[1].isupper():
        s = s[1:]

    # Handle eclipsis prefixes (must check before lenition)
    eclipsis_map = {
        'mb': 'B', 'gc': 'C', 'nd': 'D',
        'bhf': 'F', 'ng': 'G', 'bp': 'P', 'dt': 'T',
        'mB': 'B', 'gC': 'C', 'nD': 'D',
        'bhF': 'F', 'nG': 'G', 'bP': 'P', 'dT': 'T'
    }
    for prefix, canonical_initial in eclipsis_map.items():
        if s.startswith(prefix):
            remainder = s[len(prefix):]
            return canonical_initial + remainder

    # Handle vowel eclipsis: n- prefix before vowels
    if len(s) > 2 and s[0] == 'n' and s[1] == '-':
        return s[2:].capitalize()
    if len(s) > 1 and s[0] == 'n' and s[1].isupper():
        return s[1:]

    # Handle h- prefix before vowels: hÉireann → Éireann
    if len(s) > 1 and s[0] == 'h' and s[1].isupper():
        return s[1:]
    if len(s) > 2 and s[:2] == 'h-':
        return s[2:].capitalize()

    # Handle lenition: consonant + h → consonant
    lenition_map = {
        'Bh': 'B', 'Ch': 'C', 'Dh': 'D', 'Fh': 'F',
        'Gh': 'G', 'Mh': 'M', 'Ph': 'P', 'Sh': 'S', 'Th': 'T',
        'bh': 'b', 'ch': 'c', 'dh': 'd', 'fh': 'f',
        'gh': 'g', 'mh': 'm', 'ph': 'p', 'sh': 's', 'th': 't'
    }
    for prefix, canonical_initial in lenition_map.items():
        if s.startswith(prefix) and len(s) > 2:
            remainder = s[len(prefix):]
            return canonical_initial + remainder

    # Handle genitive article stripping: d' before vowels
    if len(s) > 2 and s[:2] == "d'" and s[2].isupper():
        return s[2:]

    return s

# Test against known corpus cases
test_cases = [
    ('Chomhairle Cathrach Bhaile Átha Cliath', 'Comhairle Cathrach Bhaile Átha Cliath'),
    ('Pharlaimint na hEorpa', 'Parlaimint na hEorpa'),
    ('Chonradh na Gaeilge', 'Conradh na Gaeilge'),
    ('mBaile Átha Cliath', 'Baile Átha Cliath'),
    ('gCeathrú Rua', 'Ceathrú Rua'),
    ('nGaoth Dobhair', 'Gaoth Dobhair'),
    ("d'Údarás na Gaeltachta", 'Údarás na Gaeltachta'),
    ('hÉireann', 'Éireann'),
    ('Aontais Eorpaigh', 'Aontais Eorpaigh'),
]

print("Canonicalisation test results:")
for surface, expected in test_cases:
    result = canonicalise_irish(surface)
    status = 'OK' if result == expected else 'Error'
    print(f"{status} '{surface}' → '{result}' (expected '{expected}')")

Canonicalisation test results:
OK 'Chomhairle Cathrach Bhaile Átha Cliath' → 'Comhairle Cathrach Bhaile Átha Cliath' (expected 'Comhairle Cathrach Bhaile Átha Cliath')
OK 'Pharlaimint na hEorpa' → 'Parlaimint na hEorpa' (expected 'Parlaimint na hEorpa')
OK 'Chonradh na Gaeilge' → 'Conradh na Gaeilge' (expected 'Conradh na Gaeilge')
OK 'mBaile Átha Cliath' → 'Baile Átha Cliath' (expected 'Baile Átha Cliath')
OK 'gCeathrú Rua' → 'Ceathrú Rua' (expected 'Ceathrú Rua')
OK 'nGaoth Dobhair' → 'Gaoth Dobhair' (expected 'Gaoth Dobhair')
OK 'd'Údarás na Gaeltachta' → 'Údarás na Gaeltachta' (expected 'Údarás na Gaeltachta')
OK 'hÉireann' → 'Éireann' (expected 'Éireann')
OK 'Aontais Eorpaigh' → 'Aontais Eorpaigh' (expected 'Aontais Eorpaigh')


## Wikidata Entity Fetch: PER Nodes

Full Wikidata entity JSON was fetched for 1,295 confident PER QIDs via the 
REST API (Special:EntityData). The SPARQL endpoint was unavailable throughout 
Phase 2 — all Wikidata access used the REST API exclusively.

**Note:** This cell is a documented record. The fetch took several hours and required rate limiting. 
Output files are available in the `irish-ner-kg-consolidated` Kaggle dataset at `michaelmarkey64/irish-ner-kg-consolidated` under `kg/phase_a/`.

### Field Completion

| Field | Count | Rate | Notes |
|---|---|---|---|
| PER nodes saved | 1,295 | 100% 
| Irish labels (label_ga) | 827 | 63.9% | Many minor TDs absent from Wikidata |
| party_qid filled | 788 | 60.8% | Independents and historical members have none |
| position_qids filled | 1,182 | 91.3% | Rich connectivity for PyKEEN |
| country_qid filled | 1,220 | 94.2% | Solid |
| constituency_qid | 0 | 0% | P768 not populated in Wikidata for most Irish politicians |
| oireachtas_id | 0 | 0% | P6823 sparsely populated — member_code used instead |

## Wikidata Entity Fetch: ORG and LOC Nodes

### ORG Nodes

Full Wikidata entity JSON fetched for 142 confident ORG QIDs via REST API.

| Field | Count | Rate | Notes |
|---|---|---|---|
| ORG nodes saved | 142 | 100% | Zero errors |
| Irish labels (label_ga) | 123 | 86.6% | Strong signal — QID matching is high precision |
| HQ links (hq_qid) | 71 | 50% | Abstract/political orgs have no physical HQ in Wikidata |

### LOC Nodes

Full Wikidata entity JSON fetched for 243 confident LOC QIDs via REST API.

| Field | Count | Rate | Notes |
|---|---|---|---|
| LOC nodes saved | 243 | 100% | Zero errors |
| Logainm IDs | partial | — | P6872 sparsely populated — Logainm API integration deferred to Phase 3 |

### Second Pass on Uncertain Matches

A second pass was run on all uncertain QIDs with tighter confirmation filters:
P27=Q27 (Irish citizenship) AND at least one Oireachtas-related position in 
P39 for PER; P17=Q27 (located in Ireland) for LOC.

| Pass | QIDs checked | Newly confirmed | Yield |
|---|---|---|---|
| PER second pass | 396 | 5 | 1.3% |
| LOC second pass | 125 | 14 | 11.2% |

The 1.3% PER yield confirms the first-pass confidence filter was already high 
precision. The 391 rejected uncertain PER QIDs were incorrectly matched QIDs — 
Irish diaspora politicians, name collisions, and historical figures.

**Updated node counts after second pass:** PER: 1,300 | LOC: 257 | ORG: 142

## Coverage Diagnostic

Before building stub nodes, a coverage diagnostic was run against the full 
Adkins corpus entity lists. This revealed two important findings.

### Finding 1 — PER Surface Form Variation

The Adkins corpus PER entities are Irish-language surface forms, not full 
English names. Many are surname-only (Humphreys, Ó Cuív), first-name-only 
(Noel, Gráinne), or title references (Taoiseach, Aire Sláinte). These cannot 
be matched to unique Wikidata nodes regardless of KG completeness. This is likely 
a property of Irish political discourse rather than a gap in Wikidata coverage

### Finding 2 — Role References Misclassified as PER

33 corpus entities initially classified as PER were ministerial titles or role 
references (e.g. Aire Turasóireachta, Cultúir, Ealaíon, Gaeltachta, Spóirt 
agus Meán). A further 9 were corpus noise (fictional characters and common 
nouns from literary texts: Van Helsing, An Pionta, Tommy an Táilliúra). These 
were reclassified before stub creation.

**Adjusted corpus denominator:** 651 raw PER entities → 585 after removing 
role references → 618 named individuals

### Pre-Stub Wikidata-Grounded Coverage

| Type | Corpus entities | Matched | Rate | Match method |
|---|---|---|---|---|
| PER | 618 named | 91 | 15.6% | Exact + substring |
| LOC | 573 | 190 | 33.2% | Exact string |
| ORG | 656 | 112 | 17.1% | Exact string |
| Total | 1,814 (adj.) | 393 | 21.7% | — |

The 15.6% PER match rate reflects Irish discourse convention of surname and 
title reference rather than KG incompleteness - this requires explicit 
treatment in the dissertation methodology.

In [24]:
# Node CSVs produced by the Wikidata fetch pipeline.
# Available at michaelmarkey64/irish-ner-kg-consolidated under kg/phase_a/ on Kaggle.
# per_nodes.csv  — Wikidata-grounded PER nodes
# loc_nodes.csv  — Wikidata-grounded LOC nodes
# org_nodes.csv  — Wikidata-grounded ORG nodes
# stub_nodes.csv — ungrounded entities; entity string and type only, excluded from PyKEEN embedding training

per  = pd.read_csv(KG_BASE + 'per_nodes.csv')
loc  = pd.read_csv(KG_BASE + 'loc_nodes.csv')
org  = pd.read_csv(KG_BASE + 'org_nodes.csv')
stub = pd.read_csv(KG_BASE + 'stub_nodes.csv')

print("Node file shapes:")
print(f"  PER:  {per.shape}")
print(f"  LOC:  {loc.shape}")
print(f"  ORG:  {org.shape}")
print(f"  Stub: {stub.shape}")

print("\nPER columns:",  per.columns.tolist())
print("LOC columns:",  loc.columns.tolist())
print("ORG columns:",  org.columns.tolist())
print("Stub columns:", stub.columns.tolist())

print("\nStub node type breakdown:")
print(stub['node_type'].value_counts())

Node file shapes:
  PER nodes:  (1300, 13)
  LOC nodes:  (257, 10)
  ORG nodes:  (142, 10)
  Stub nodes: (1487, 7)

PER columns: ['qid', 'label_en', 'label_ga', 'description_en', 'party_qid', 'position_qids', 'constituency_qid', 'country_qid', 'oireachtas_id', 'date_of_birth', 'full_name_en', 'party_name', 'constituency_name']
LOC columns: ['qid', 'label_en', 'label_ga', 'description_en', 'instance_of_qids', 'parent_qid', 'country_qid', 'logainm_id', 'entity', 'canonical']
ORG columns: ['qid', 'label_en', 'label_ga', 'description_en', 'instance_of_qids', 'hq_qid', 'country_qid', 'parent_org_qid', 'entity', 'canonical']
Stub columns: ['stub_id', 'entity', 'node_type', 'stub_class', 'qid', 'label_en', 'label_ga']

Stub node type breakdown:
node_type
PER    560
ORG    544
LOC    383
Name: count, dtype: int64


## Stub Node Creation

Stub nodes were created for all corpus entities not matched to a Wikidata QID. 
Stubs carry the entity string and type but no Wikidata properties and are 
excluded from PyKEEN embedding training. This ensures 100% corpus coverage in 
the graph while keeping the embedding subgraph clean.

### Stub Node Taxonomy

| Type | Class | Count | Notes |
|---|---|---|---|
| PER | NAMED_INDIVIDUAL | 506 | Named persons with no Wikidata match |
| PER | ROLE_REFERENCE | 45 | Ministerial and parliamentary titles |
| PER | CORPUS_NOISE | 9 | Excluded from graph entirely |
| LOC | NAMED_LOCATION | 383 | — |
| ORG | NAMED_ORG | 544 | — |
| Total loaded | | **1,478** | 9 noise entities excluded |

## Neo4j Graph Load

**Note:** This section is a documented record. The Neo4j instance runs locally 
and cannot be reproduced on Kaggle.

- Instance name: `dissertation-kg-phase2`
- Neo4j version: 2026.04.0
- URI: `bolt://localhost:7687`
- Python driver: `neo4j` v6.2.0

All nodes were loaded with MERGE on `wikidata_id` to prevent duplicates. 
Constraints and indexes were created on `node_id` for all three main node types.
HOLDS_POSITION edges were loaded via batch UNWIND — row-by-row session approach 
caused SessionError on large datasets.

### Final Graph Node Counts

| Label | CSV count | Graph count | Diff | Reason |
|---|---|---|---|---|
| Person | 1,300 | 1,272 | -28 | Duplicate QIDs collapsed by MERGE |
| Location | 257 | 185 | -72 | Duplicate QIDs collapsed |
| Organisation | 142 | 112 | -30 | Duplicate QIDs collapsed |
| Office | — | 266 | new | Created from position_qids strings |
| StubNode | 1,478 | 1,478 | 0 
| Total | | **3,313** | | |

### Edge Counts

| Relation | Count | Notes |
|---|---|---|
| MEMBER_OF (PER→ORG) | 588 | Via party_qid |
| HOLDS_POSITION (PER→Office) | 2,551 | Deduplicated from 5,757 raw pairs |
| LOCATED_IN (LOC→LOC) | 84 | Via parent_qid — sparse in Wikidata for Irish locations |
| HQ_IN (ORG→LOC) | 32 | 71 HQ links in CSV, 32 matched to existing LOC nodes |
| REPRESENTS (PER→LOC) | 28 | County-level match only |
| **Total** | **3,283** | |

## Graph Validation

| Metric | Value | Notes |
|---|---|---|
| Wikidata-grounded nodes | 1,569 (47.4%) | — |
| Stub nodes | 1,478 (44.6%) | — |
| Office nodes (derived) | 266 (8.0%) | Created from position claims |
| Total nodes | 3,313 | — |
| Total edges | 3,283 | — |
| Edge density (whole graph) | 0.99 per node | Low due to zero-edge stub nodes |
| Edge density (grounded only) | 2.09 per node | Embedding-relevant subgraph |
| Connected Person nodes | 1,167 / 1,272 | 91.7% |
| Pre-stub corpus coverage | 21.7% | 393 / 1,814 named entities |
| Post-stub corpus coverage | 100% | By definition |

The edge density difference between the whole graph (0.99) and the grounded 
subgraph (2.09) is important for interpreting PyKEEN results — the embedding 
model only trains on the grounded subgraph, which has meaningful but still 
sparse connectivity.

## PyKEEN Embedding Training

**Note:** This section is a documented record. Training ran locally on CPU and took several hours. 
The output embeddings are available in the `irish-ner-kg-consolidated` Kaggle dataset at `michaelmarkey64/irish-ner-kg-consolidated` 
under `kg/phase_a/`.

The KG was exported as a triples TSV (subject, predicate, object) containing 
3,283 triples from grounded nodes only — stub nodes excluded. Three models were 
trained via PyKEEN:

### Embedding Results

| Model | MRR | Hits@1 | Hits@10 |
|---|---|---|---|
| TransE | 0.130 | 0.000 | 0.319 |
| RotatE | 0.061 | 0.021 | 0.147 |
| ComplEx | 0.006 | 0.002 | 0.009 |

**Selected model: TransE.** TransE achieved the strongest performance across 
all metrics, consistent with prior findings that TransE is competitive on small, 
sparse graphs where more expressive models lack sufficient training signal. 
ComplEx produced near-random results (MRR 0.006), likely attributable to limited graph 
size relative to model parameter requirements.

The pronounced tail-prediction advantage (Hits@10 tail = 0.608 vs head = 0.030) 
reflects the hub-and-spoke structure of the graph, where many Person nodes share 
a small set of Organisation and Location targets.

### Output Files

| File | Description |  |
|---|---|---|
| `TransE_qid_embeddings.pkl` | 1,563 × 128-dim vectors keyed by QID 
| `TransE_node_id_embeddings.pkl` | Full node ID keyed version 
| `kg_triples.tsv` | 3,283 triples (subject, predicate, object) 

TransE embeddings are loaded in Notebook 2 for injection into gaBERT-CRF at 
training time.

## References

Adkins, J., Cassidy, L., Laoide-Kemp, D., & Nicolai, G. (2025). *ner4Irish: 
A named entity recognition dataset for Irish*. GitHub. 
https://github.com/janeadkinspgr/ner4Irish

Lynn, T., & Foster, J. (2016). Universal dependencies for Irish. In 
*Proceedings of the Celtic Language Technology Workshop* (pp. 79–92). 
Association for Computational Linguistics.

Lynn, T., Walsh, P., & Foster, J. (2019). Minority language Twitter: Part-of-speech 
tagging and analysis of Irish tweets. In *Proceedings of the 3rd Workshop on 
Computational Approaches to Linguistic Code-Switching* (pp. 1–10). Association 
for Computational Linguistics. https://doi.org/10.18653/v1/W19-1001

Scannell, K. (2014). *An caighdeánóir: Automatic standardisation of Irish text*. 
Fiontar, Dublin City University.

Shi, B., & Weninger, T. (2018). Open-world knowledge graph completion. In 
*Proceedings of the 32nd AAAI Conference on Artificial Intelligence* 
(pp. 1957–1964). AAAI Press.

Ali, M., Berrendorf, M., Hoyt, C. T., Vermue, L., Sharifzadeh, S., Tresp, V., 
& Lehmann, J. (2021). PyKEEN 1.0: A Python library for training and evaluating 
knowledge graph embeddings. *Journal of Machine Learning Research*, *22*(82), 
1–6. http://jmlr.org/papers/v22/20-1168.html

Bordes, A., Usunier, N., Garcia-Duran, A., Weston, J., & Yakhnenko, O. (2013). 
Translating embeddings for modeling multi-relational data. In 
*Advances in Neural Information Processing Systems* (Vol. 26, pp. 2787–2795). 
Curran Associates.

Devlin, J., Chang, M.-W., Lee, K., & Toutanova, K. (2019). BERT: Pre-training 
of deep bidirectional transformers for language understanding. In 
*Proceedings of the 2019 Conference of the North American Chapter of the 
Association for Computational Linguistics: Human Language Technologies* 
(Vol. 1, pp. 4171–4186). Association for Computational Linguistics. 
https://doi.org/10.18653/v1/N19-1423

De Brún, A., & Lynn, T. (2022). *Gabert: Pre-trained BERT for the Irish 
language*. DCU-NLP, Dublin City University. 
https://huggingface.co/DCU-NLP/bert-base-irish-cased-v1

Vrandečić, D., & Krötzsch, M. (2014). Wikidata: A free collaborative 
knowledgebase. *Communications of the ACM*, *57*(10), 78–85. 
https://doi.org/10.1145/2629489